No,３

In [1]:
import os
import re
import tkinter as tk
from tkinter import filedialog

末端フォルダ内のファイル数を調整：設定したファイル数を超える場合のみ適用する。削除するファイルは昇順

NR用

In [4]:


def main():
    # 1. フォルダ選択ダイアログの表示設定
    root = tk.Tk()
    root.withdraw() # メインウィンドウを非表示
    root.attributes('-topmost', True) # ダイアログが裏に隠れないように最前面へ
    
    print("ダイアログから大元のフォルダを選択してください...")
    base_dir = filedialog.askdirectory(title="大元のフォルダを選択してください")
    
    if not base_dir:
        print("フォルダが選択されませんでした。処理を中断します。")
        return

    print(f"選択されたフォルダ: {base_dir}")
    print("-" * 30)

    # 2. ファイル名のルールを正規表現で定義
    # ^([^$]+) : $以外の文字列（特定の数字）
    # \$       : 区切り文字の $
    # (\d+)    : 連番（数字）
    # (.*)$    : 拡張子など（あれば）
    pattern = re.compile(r'^([^$]+)\$(\d+)(.*)$')
    
    deleted_total = 0
    
    # 3. フォルダ内の探索と削除処理
    for dirpath, dirnames, filenames in os.walk(base_dir):
        target_files = [] # 条件に合うファイルを格納するリスト
        
        # フォルダ内のファイルを確認
        for filename in filenames:
            match = pattern.match(filename)
            if match:
                # 連番部分を取り出して整数(int)に変換
                seq_num = int(match.group(2))
                # (ファイル名, 連番の数値) のセットでリストに追加
                target_files.append((filename, seq_num))
            else:
                # ルールに合致しないファイルはスキップ（安全対策）
                pass
                
        # 該当ファイルが10個より多い場合のみ削除処理を実行
        if len(target_files) > 10:
            # 連番の数値が小さい順（昇順）にソート
            target_files.sort(key=lambda x: x[1])
            
            # 10個残すために、削除すべきファイルの数を計算
            delete_count = len(target_files) - 10
            
            # 連番が小さい方から順に削除
            for i in range(delete_count):
                file_to_delete = target_files[i][0]
                file_path = os.path.join(dirpath, file_to_delete)
                
                try:
                    os.remove(file_path)
                    print(f"削除しました: {file_path}")
                    deleted_total += 1
                except Exception as e:
                    print(f"削除エラー ({file_path}): {e}")

    print("-" * 30)
    print(f"処理が完了しました。合計 {deleted_total} 個のファイルを削除しました。")

# 実行
if __name__ == "__main__":
    main()

ダイアログから大元のフォルダを選択してください...
選択されたフォルダ: E:/001_2026AI班/futabaデータ/半透明PP(J707EG)/合わせたやつ(0.5mm_再々,1.0mm_＋再)_/csv
------------------------------
------------------------------
処理が完了しました。合計 0 個のファイルを削除しました。


Futaba用

In [5]:
import os
import re
import tkinter as tk
from tkinter import filedialog

def main():
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    
    print("ダイアログから大元のフォルダを選択してください...")
    base_dir = filedialog.askdirectory(title="大元のフォルダを選択してください")
    
    if not base_dir:
        print("フォルダが選択されませんでした。処理を中断します。")
        return

    print(f"選択されたフォルダ: {base_dir}")
    print("-" * 30)

    # 【修正】日付と時間を無視するための正規表現
    # ^(.*)    : 任意の文字列 (例: "MPS5 settei") -> group(1)として取得
    # _(\d{8}) : _ と 8桁の日付 (例: "_20260618")
    # _(\d{6}) : _ と 6桁の時間 (例: "_121532")
    # _(\d+)   : _ と 連番 (例: "_000001")      -> group(4)として取得
    # (.*)$    : 拡張子など (例: ".csv")
    pattern = re.compile(r'^(.*)_(\d{8})_(\d{6})_(\d+)(.*)$')
    
    deleted_total = 0
    
    for dirpath, dirnames, filenames in os.walk(base_dir):
        target_groups = {} 
        
        for filename in filenames:
            match = pattern.match(filename)
            if match:
                # group(1)には "MPS5 settei" のような大元の名前だけが入る
                prefix = match.group(1)       
                # group(4)には "000001" のような連番だけが入る
                seq_num = int(match.group(4)) 
                
                if prefix not in target_groups:
                    target_groups[prefix] = []
                
                target_groups[prefix].append((filename, seq_num))
            else:
                # 想定外のファイル名があった場合のみスキップ表示
                print(f"【スキップ】ルール不一致: {filename}")
                
        # グループごとに削除判定
        for prefix, files in target_groups.items():
            if len(files) > 10:
                files.sort(key=lambda x: x[1])
                delete_count = len(files) - 10
                
                for i in range(delete_count):
                    file_to_delete = files[i][0]
                    file_path = os.path.join(dirpath, file_to_delete)
                    
                    try:
                        os.remove(file_path)
                        print(f"削除しました: {file_path}")
                        deleted_total += 1
                    except Exception as e:
                        print(f"削除エラー ({file_path}): {e}")

    print("-" * 30)
    print(f"処理が完了しました。合計 {deleted_total} 個のファイルを削除しました。")

if __name__ == "__main__":
    main()

ダイアログから大元のフォルダを選択してください...
選択されたフォルダ: E:/001_2026AI班/futabaデータ/黒色PP(J707EG＋カーボンブラック)
------------------------------
削除しました: E:/001_2026AI班/futabaデータ/黒色PP(J707EG＋カーボンブラック)\0.5mm\190℃\020\MPS5 settei_20260609_150131_000001.csv
削除しました: E:/001_2026AI班/futabaデータ/黒色PP(J707EG＋カーボンブラック)\0.5mm\190℃\020\MPS5 settei_20260609_150158_000002.csv
削除しました: E:/001_2026AI班/futabaデータ/黒色PP(J707EG＋カーボンブラック)\0.5mm\190℃\020\MPS5 settei_20260609_150226_000003.csv
削除しました: E:/001_2026AI班/futabaデータ/黒色PP(J707EG＋カーボンブラック)\0.5mm\190℃\020\MPS5 settei_20260609_150253_000004.csv
削除しました: E:/001_2026AI班/futabaデータ/黒色PP(J707EG＋カーボンブラック)\0.5mm\190℃\020\MPS5 settei_20260609_150321_000005.csv
削除しました: E:/001_2026AI班/futabaデータ/黒色PP(J707EG＋カーボンブラック)\0.5mm\190℃\020\MPS5 settei_20260609_150348_000006.csv
削除しました: E:/001_2026AI班/futabaデータ/黒色PP(J707EG＋カーボンブラック)\0.5mm\190℃\020\MPS5 settei_20260609_150416_000007.csv
削除しました: E:/001_2026AI班/futabaデータ/黒色PP(J707EG＋カーボンブラック)\0.5mm\190℃\040\MPS5 settei_20260609_151049_000001.csv
削除しました: E:/0